# Buy the Panic: Market-Wide Mean Reversion on MOEX

Testing whether broad market selloffs (measured by index magnitude and breadth of decline) predict short-term reversals. Combines IMOEX drawdown thresholds with breadth indicators (fraction of stocks declining) to generate panic signals, then evaluates forward returns across different portfolio construction methods and holding periods.

In [ ]:
# Load daily stock and index data, compute returns.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

from data_cache import get_stocks, get_index, RU_UNIVERSE_TOP20, RU_UNIVERSE_100, MOEX_INDEX

COMMISSION_PCT = 0.0005
SLIPPAGE_PCT = 0.0008
TOTAL_COST_PCT = (COMMISSION_PCT + SLIPPAGE_PCT) * 2

print("Loading data...")
stocks_d = get_stocks(RU_UNIVERSE_100, interval=24, start="2022-01-01", verbose=False)
stocks_d["date"] = stocks_d["timestamp"].dt.date

prices = stocks_d.pivot_table(index="date", columns="ticker", values="close", aggfunc="last")
prices.index = pd.to_datetime(prices.index)
prices = prices.sort_index()
daily_ret = prices.pct_change()

imoex = get_index(MOEX_INDEX, interval=24, start="2022-01-01", verbose=False)
if not imoex.empty:
    imoex["date"] = pd.to_datetime(imoex["timestamp"].dt.date)
    imoex_close = imoex.set_index("date")["close"].sort_index()
    imoex_ret = imoex_close.pct_change()
    print(f"IMOEX: {len(imoex_close)} daily bars, {imoex_close.index.min().date()} to {imoex_close.index.max().date()}")
else:
    print("IMOEX not loaded, using equal-weight portfolio as proxy")
    imoex_close = (prices.fillna(method='ffill') / prices.fillna(method='ffill').iloc[0]).mean(axis=1)
    imoex_ret = imoex_close.pct_change()

print(f"Stocks: {len(prices)} daily bars, {prices.shape[1]} tickers")

In [ ]:
# Compute panic signals: magnitude (IMOEX drawdown), breadth (fraction of stocks declining), and combo.
breadth = (daily_ret < 0).sum(axis=1) / daily_ret.notna().sum(axis=1)

imoex_ret_1d = imoex_ret
imoex_ret_3d = imoex_close.pct_change(3)
imoex_ret_5d = imoex_close.pct_change(5)

print("IMOEX daily return distribution:")
print(imoex_ret_1d.describe().round(4))

print(f"\nDays with IMOEX <-1%: {(imoex_ret_1d < -0.01).sum()}")
print(f"Days with IMOEX <-2%: {(imoex_ret_1d < -0.02).sum()}")
print(f"Days with IMOEX <-3%: {(imoex_ret_1d < -0.03).sum()}")

print(f"\nBreadth distribution (fraction of declining stocks per day):")
print(breadth.describe().round(3))

print(f"\nDays with breadth>0.6: {(breadth > 0.6).sum()}")
print(f"Days with breadth>0.75: {(breadth > 0.75).sum()}")
print(f"Days with breadth>0.85: {(breadth > 0.85).sum()}")

combo_score = (-imoex_ret_1d * breadth).dropna()
top_panic = combo_score.nlargest(10)
print(f"\nTop-10 panic days (magnitude x breadth):")
panic_summary = pd.DataFrame({
    "imoex_ret_pct": imoex_ret_1d.loc[top_panic.index] * 100,
    "breadth": breadth.loc[top_panic.index],
}).round(3)
display(panic_summary)

In [ ]:
# Measure forward IMOEX returns after panic signals vs baseline, with t-tests.
HORIZONS = [1, 3, 5, 10, 20]

forward_imoex = pd.DataFrame(index=imoex_close.index)
for h in HORIZONS:
    forward_imoex[f"fwd_d{h}"] = imoex_close.shift(-h) / imoex_close - 1


def signal_analysis(signal_dates, forward_df, baseline_df, label):
    sig_fwd = forward_df.loc[forward_df.index.isin(signal_dates)]
    if sig_fwd.empty:
        return None
    
    rows = []
    for h in HORIZONS:
        col = f"fwd_d{h}"
        sig_ret = sig_fwd[col].dropna()
        base_ret = baseline_df[col].dropna()
        if len(sig_ret) < 3:
            continue
        t_stat, p_val = stats.ttest_ind(sig_ret, base_ret)
        rows.append({
            "horizon_d": h,
            "n_signals": len(sig_ret),
            "panic_mean_pct": sig_ret.mean() * 100,
            "baseline_mean_pct": base_ret.mean() * 100,
            "edge_pct": (sig_ret.mean() - base_ret.mean()) * 100,
            "panic_winrate": (sig_ret > 0).mean() * 100,
            "baseline_winrate": (base_ret > 0).mean() * 100,
            "t_stat": t_stat,
            "p_value": p_val,
        })
    
    df = pd.DataFrame(rows)
    print(f"\n=== {label} ===")
    print(f"Signals: {len(sig_fwd)}")
    display(df.round(3))
    return df


print("Forward IMOEX returns after different panic signals:\n")

sig1 = imoex_ret_1d[imoex_ret_1d < -0.015].index
signal_analysis(sig1, forward_imoex, forward_imoex, "MAGNITUDE: IMOEX <-1.5% daily")

sig2 = imoex_ret_1d[imoex_ret_1d < -0.03].index
signal_analysis(sig2, forward_imoex, forward_imoex, "MAGNITUDE: IMOEX <-3% daily")

sig3 = breadth[breadth > 0.75].index
signal_analysis(sig3, forward_imoex, forward_imoex, "BREADTH: >=75% stocks declining")

sig4 = breadth[breadth > 0.85].index
signal_analysis(sig4, forward_imoex, forward_imoex, "BREADTH: >=85% stocks declining")

combo_dates = imoex_ret_1d[(imoex_ret_1d < -0.015) & (breadth.reindex(imoex_ret_1d.index) > 0.75)].index
signal_analysis(combo_dates, forward_imoex, forward_imoex, "COMBO: IMOEX <-1.5% + breadth >=75%")

combo_strict = imoex_ret_1d[(imoex_ret_1d < -0.02) & (breadth.reindex(imoex_ret_1d.index) > 0.80)].index
signal_analysis(combo_strict, forward_imoex, forward_imoex, "COMBO STRICT: IMOEX <-2% + breadth >=80%")

In [ ]:
# Backtest: compare portfolio types (top10, worst performers, all) across signal configs and hold periods.
def backtest_buy_panic(prices_df, signal_dates, hold_days=10,
                        portfolio_type="top10", top_n=5,
                        cost_pct=TOTAL_COST_PCT):
    trades = []
    available_tickers = [t for t in RU_UNIVERSE_TOP20[:10] if t in prices_df.columns]
    
    for signal_date in signal_dates:
        try:
            entry_idx = prices_df.index.get_loc(signal_date)
        except KeyError:
            continue
        
        if entry_idx + hold_days >= len(prices_df):
            continue
        
        exit_date = prices_df.index[entry_idx + hold_days]
        entry_prices = prices_df.iloc[entry_idx]
        exit_prices = prices_df.iloc[entry_idx + hold_days]
        
        if portfolio_type == "top10":
            picks = available_tickers
        elif portfolio_type == "worst":
            day_returns = daily_ret.loc[signal_date].dropna()
            picks = day_returns.nsmallest(top_n).index.tolist()
        elif portfolio_type == "all":
            picks = [t for t in prices_df.columns if not pd.isna(entry_prices[t])]
        else:
            continue
        
        rets = []
        for ticker in picks:
            if ticker not in entry_prices.index or pd.isna(entry_prices[ticker]):
                continue
            if ticker not in exit_prices.index or pd.isna(exit_prices[ticker]):
                continue
            r = exit_prices[ticker] / entry_prices[ticker] - 1
            rets.append(r)
        
        if not rets:
            continue
        
        port_ret = np.mean(rets) - cost_pct
        trades.append({
            "signal_date": signal_date,
            "exit_date": exit_date,
            "n_picks": len(rets),
            "port_ret": port_ret,
        })
    
    return pd.DataFrame(trades)


def trade_metrics(trades_df, hold_days):
    if trades_df.empty or len(trades_df) < 3:
        return None
    
    rets = trades_df["port_ret"]
    n = len(rets)
    
    if len(trades_df) >= 2:
        date_span_days = (trades_df["signal_date"].max() - trades_df["signal_date"].min()).days
        avg_days_between = date_span_days / max(n - 1, 1)
        signals_per_year = 365 / max(avg_days_between, 1)
    else:
        signals_per_year = 252 / hold_days
    
    avg_ret = rets.mean()
    std_ret = rets.std() if len(rets) > 1 else 0
    
    ann_ret_simple = avg_ret * signals_per_year
    sharpe = avg_ret / std_ret * np.sqrt(signals_per_year) if std_ret > 0 else 0
    
    cum = (1 + rets.values).cumprod()
    max_dd = (np.maximum.accumulate(cum) - cum) / np.maximum.accumulate(cum)
    
    return {
        "trades": n,
        "winrate_pct": (rets > 0).mean() * 100,
        "avg_ret_pct": avg_ret * 100,
        "median_ret_pct": rets.median() * 100,
        "std_ret_pct": std_ret * 100,
        "best_pct": rets.max() * 100,
        "worst_pct": rets.min() * 100,
        "sharpe_ann": sharpe,
        "ann_ret_pct": ann_ret_simple * 100,
        "max_dd_pct": max_dd.max() * 100,
        "signals_per_year": signals_per_year,
    }


signal_configs = [
    ("magnitude_-1.5%", imoex_ret_1d[imoex_ret_1d < -0.015].index),
    ("magnitude_-2%",   imoex_ret_1d[imoex_ret_1d < -0.020].index),
    ("magnitude_-3%",   imoex_ret_1d[imoex_ret_1d < -0.030].index),
    ("breadth_75%",     breadth[breadth > 0.75].index),
    ("breadth_85%",     breadth[breadth > 0.85].index),
    ("combo_-1.5%/75%", imoex_ret_1d[(imoex_ret_1d < -0.015) & (breadth.reindex(imoex_ret_1d.index) > 0.75)].index),
    ("combo_-2%/80%",   imoex_ret_1d[(imoex_ret_1d < -0.020) & (breadth.reindex(imoex_ret_1d.index) > 0.80)].index),
]

results = []
for sig_name, sig_dates in signal_configs:
    if len(sig_dates) < 3:
        continue
    for hold in [3, 5, 10, 20]:
        for port_type in ["top10", "worst", "all"]:
            trades = backtest_buy_panic(prices, sig_dates,
                                          hold_days=hold,
                                          portfolio_type=port_type)
            m = trade_metrics(trades, hold)
            if m is None:
                continue
            m["signal"] = sig_name
            m["hold_days"] = hold
            m["portfolio"] = port_type
            results.append(m)

results_df = pd.DataFrame(results).sort_values("sharpe_ann", ascending=False)
print(f"Top-15 configurations by Sharpe:")
cols = ["signal", "portfolio", "hold_days", "trades", "winrate_pct",
        "avg_ret_pct", "ann_ret_pct", "sharpe_ann", "max_dd_pct"]
display(results_df[cols].head(15).round(3))

print(f"\nBottom-5:")
display(results_df[cols].tail(5).round(3))

In [ ]:
# Equity curve and PnL distribution for the best configuration.
if not results_df.empty:
    best = results_df.iloc[0]
    print(f"Best configuration:")
    print(f"  Signal: {best['signal']}")
    print(f"  Portfolio: {best['portfolio']}")
    print(f"  Hold: {best['hold_days']:.0f} days")
    print(f"  Sharpe: {best['sharpe_ann']:.2f} | Annual: {best['ann_ret_pct']:+.1f}%")
    print(f"  Trades: {best['trades']:.0f} (~{best['signals_per_year']:.0f}/year)")
    
    sig_name = best["signal"]
    sig_dates = dict(signal_configs)[sig_name]
    best_trades = backtest_buy_panic(prices, sig_dates,
                                       hold_days=int(best["hold_days"]),
                                       portfolio_type=best["portfolio"])
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    cum = (1 + best_trades["port_ret"]).cumprod() * 100 - 100
    axes[0,0].plot(best_trades["signal_date"], cum, "o-", color="#2ecc71",
                    linewidth=1.5, markersize=8)
    axes[0,0].axhline(0, color="black", linewidth=0.5)
    axes[0,0].set_title(f"Equity curve: {sig_name} / {best['portfolio']}, hold {int(best['hold_days'])}d")
    axes[0,0].set_ylabel("Cumulative return, %")
    
    colors_t = ["#2ecc71" if x > 0 else "#e74c3c" for x in best_trades["port_ret"]]
    axes[0,1].bar(range(len(best_trades)), best_trades["port_ret"] * 100, color=colors_t, alpha=0.8)
    axes[0,1].axhline(0, color="black", linewidth=0.5)
    axes[0,1].axhline(best["avg_ret_pct"], color="blue", linestyle="--", alpha=0.5,
                       label=f"Avg: {best['avg_ret_pct']:.2f}%")
    axes[0,1].set_title(f"Trade PnL (winrate {best['winrate_pct']:.0f}%)")
    axes[0,1].set_ylabel("Trade PnL, %")
    axes[0,1].legend()
    
    axes[1,0].hist(best_trades["port_ret"] * 100, bins=15, color="steelblue",
                    edgecolor="black", alpha=0.7)
    axes[1,0].axvline(0, color="black", linewidth=0.5)
    axes[1,0].axvline(best_trades["port_ret"].mean() * 100, color="green",
                       linestyle="--", label=f"Mean: {best_trades['port_ret'].mean()*100:.2f}%")
    axes[1,0].axvline(best_trades["port_ret"].median() * 100, color="orange",
                       linestyle="--", label=f"Median: {best_trades['port_ret'].median()*100:.2f}%")
    axes[1,0].set_title("Trade return distribution")
    axes[1,0].set_xlabel("Trade PnL, %")
    axes[1,0].legend()
    
    axes[1,1].plot(imoex_close.index, imoex_close, color="grey", linewidth=1, alpha=0.7,
                    label="IMOEX")
    sig_panic_dates = best_trades["signal_date"]
    sig_panic_prices = imoex_close.loc[imoex_close.index.isin(sig_panic_dates)]
    axes[1,1].scatter(sig_panic_prices.index, sig_panic_prices,
                       color="red", s=80, zorder=5, label="Panic signal")
    axes[1,1].set_title(f"Panic signals on IMOEX ({len(sig_panic_dates)} events)")
    axes[1,1].set_ylabel("IMOEX")
    axes[1,1].legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("No results")

In [ ]:
# Dynamic exit: sell when price recovers a fraction of the panic-day drop, or on timeout.
def backtest_dynamic_exit(prices_df, signal_dates, max_hold_days=20,
                            portfolio_type="top10", top_n=5,
                            target_recovery=0.5, cost_pct=TOTAL_COST_PCT):
    trades = []
    available_tickers = [t for t in RU_UNIVERSE_TOP20[:10] if t in prices_df.columns]
    
    for signal_date in signal_dates:
        try:
            entry_idx = prices_df.index.get_loc(signal_date)
        except KeyError:
            continue
        
        if entry_idx + max_hold_days >= len(prices_df):
            continue
        if entry_idx == 0:
            continue
        
        pre_panic_idx = entry_idx - 1
        entry_idx_p = entry_idx
        
        if portfolio_type == "top10":
            picks = available_tickers
        elif portfolio_type == "worst":
            day_returns = daily_ret.loc[signal_date].dropna()
            picks = day_returns.nsmallest(top_n).index.tolist()
        elif portfolio_type == "all":
            picks = [t for t in prices_df.columns if not pd.isna(prices_df.iloc[entry_idx][t])]
        else:
            continue
        
        pre_prices = prices_df.iloc[pre_panic_idx]
        entry_prices = prices_df.iloc[entry_idx_p]
        
        valid_picks = []
        for t in picks:
            if pd.isna(pre_prices.get(t)) or pd.isna(entry_prices.get(t)):
                continue
            valid_picks.append(t)
        
        if not valid_picks:
            continue
        
        avg_pre = pre_prices[valid_picks].mean()
        avg_entry = entry_prices[valid_picks].mean()
        
        if avg_entry >= avg_pre:
            continue
        
        gap_back = (avg_pre - avg_entry) / avg_entry
        target_ret = target_recovery * gap_back
        
        exit_idx = None
        for h in range(1, max_hold_days + 1):
            current = prices_df.iloc[entry_idx_p + h][valid_picks].mean()
            if pd.isna(current):
                continue
            current_ret = (current - avg_entry) / avg_entry
            if current_ret >= target_ret:
                exit_idx = entry_idx_p + h
                break
        
        if exit_idx is None:
            exit_idx = entry_idx_p + max_hold_days
            exit_reason = "timeout"
        else:
            exit_reason = "target"
        
        exit_price = prices_df.iloc[exit_idx][valid_picks].mean()
        port_ret = (exit_price - avg_entry) / avg_entry - cost_pct
        
        trades.append({
            "signal_date": signal_date,
            "exit_date": prices_df.index[exit_idx],
            "hold_days": exit_idx - entry_idx_p,
            "exit_reason": exit_reason,
            "n_picks": len(valid_picks),
            "port_ret": port_ret,
        })
    
    return pd.DataFrame(trades)


print("Dynamic exit (target recovery 0.5, max hold 20d):\n")
for sig_name, sig_dates in signal_configs:
    if len(sig_dates) < 3:
        continue
    for port_type in ["top10", "worst"]:
        trades = backtest_dynamic_exit(prices, sig_dates,
                                         max_hold_days=20,
                                         portfolio_type=port_type,
                                         target_recovery=0.5)
        m = trade_metrics(trades, 10)
        if m is None:
            continue
        avg_hold = trades["hold_days"].mean()
        target_rate = (trades["exit_reason"] == "target").mean() * 100
        print(f"  {sig_name} / {port_type}: {m['trades']} trades, "
              f"WR {m['winrate_pct']:.0f}%, avg PnL {m['avg_ret_pct']:+.2f}%, "
              f"Sharpe {m['sharpe_ann']:.2f}, avg hold {avg_hold:.0f}d, "
              f"target hit {target_rate:.0f}%")

## Results

Weak mean-reversion effect overall. No signal configuration produces statistically significant positive forward returns vs baseline (all t-tests insignificant at 5% level). The best fixed-hold configuration (magnitude -2%, worst performers, 10-day hold) achieves Sharpe 0.27 with ~14% annualized return but 71% max drawdown. Dynamic exit with 50% recovery target improves win rate to 65% but does not materially change risk-adjusted returns. The effect is not robust enough for standalone deployment.